[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module4/05-eda.ipynb)

# Exploratory Data Analysis (EDA): A Full Workflow

**Module 4 — Data Science & Visualization** | Estimated time: 35 minutes

## Learning Objectives

By the end of this notebook you will be able to:
- Execute a structured EDA workflow from data loading to insights
- Summarize datasets with `info()`, `describe()`, and `value_counts()`
- Analyze distributions using histograms, KDE plots, and box plots
- Compute and visualize correlation matrices
- Identify skewness, kurtosis, and target variable relationships
- Produce a structured EDA report summary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

rng = np.random.default_rng(42)
print('Libraries loaded.')

## 1. Dataset: Synthetic Customer Churn Data

We generate a realistic tabular dataset representing customer data for a subscription service. The **target variable** is `churned` (1 = customer left, 0 = stayed).

In [ ]:
n = 800

# Features
tenure_months  = rng.integers(1, 72, n)               # how long customer has been active
monthly_charge = rng.uniform(20, 120, n).round(2)      # monthly bill
num_products   = rng.integers(1, 6, n)                 # number of subscribed products
age            = rng.integers(18, 75, n)
region         = rng.choice(['North', 'South', 'East', 'West'], n)
contract_type  = rng.choice(['Month-to-Month', 'One Year', 'Two Year'], n,
                             p=[0.55, 0.25, 0.20])
support_calls  = rng.poisson(2, n)                     # customer support interactions

# Churn probability depends on features (realistic signal)
churn_logit = (
    -2.5
    + 0.04 * monthly_charge
    - 0.05 * tenure_months
    + 0.3  * support_calls
    - 0.4  * num_products
    + np.where(contract_type == 'Month-to-Month', 1.2, 0)
    + rng.normal(0, 0.5, n)
)
churn_prob  = 1 / (1 + np.exp(-churn_logit))
churned     = (rng.uniform(0, 1, n) < churn_prob).astype(int)

df = pd.DataFrame({
    'tenure_months':  tenure_months,
    'monthly_charge': monthly_charge,
    'num_products':   num_products,
    'age':            age,
    'region':         region,
    'contract_type':  contract_type,
    'support_calls':  support_calls,
    'churned':        churned
})

# Inject a few missing values
for col, frac in [('monthly_charge', 0.03), ('age', 0.05), ('tenure_months', 0.02)]:
    mask = rng.random(n) < frac
    df.loc[mask, col] = np.nan

print(f'Dataset shape: {df.shape}')
print('\nFirst 5 rows:')
print(df.head())

## 2. Step 1 — Overview: info, head, describe

The first step of any EDA is a structural audit: how many rows, what columns exist, what types, and how many are missing.

In [ ]:
print('=== df.info() ===')
df.info()

print('\n=== df.describe() ===')
print(df.describe(include='all').T.to_string())

print('\n=== Missing values ===')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\nTotal missing: {df.isnull().sum().sum()} out of {df.size} values ({df.isnull().mean().mean():.1%})')

## 3. Step 2 — Categorical Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

categorical_cols = ['region', 'contract_type', 'churned']
for ax, col in zip(axes, categorical_cols):
    vc = df[col].value_counts()
    ax.bar(vc.index.astype(str), vc.values, color=sns.color_palette('Set2', len(vc)))
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    for i, v in enumerate(vc.values):
        ax.text(i, v + 1, str(v), ha='center', va='bottom', fontsize=9)
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Categorical Variable Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Churn rate:', df['churned'].mean().round(3))
print('\nChurn rate by contract type:')
print(df.groupby('contract_type')['churned'].mean().round(3).sort_values(ascending=False))
print('\nChurn rate by region:')
print(df.groupby('region')['churned'].mean().round(3).sort_values(ascending=False))

## 4. Step 3 — Distribution Analysis: Numerical Variables

In [ ]:
num_cols = ['tenure_months', 'monthly_charge', 'num_products', 'age', 'support_calls']

fig, axes = plt.subplots(2, len(num_cols), figsize=(18, 8))

for i, col in enumerate(num_cols):
    data = df[col].dropna()

    # Top row: histogram + KDE split by churn
    for label, color in [(0, '#4CAF50'), (1, '#F44336')]:
        subset = df[df['churned'] == label][col].dropna()
        axes[0, i].hist(subset, bins=20, alpha=0.5, color=color, density=True,
                        label=f'Churned={label}')
        subset.plot.kde(ax=axes[0, i], color=color, linewidth=2)
    axes[0, i].set_title(col)
    axes[0, i].legend(fontsize=7)
    axes[0, i].set_ylabel('Density')

    # Bottom row: boxplot by churn
    df.boxplot(column=col, by='churned', ax=axes[1, i], patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2))
    axes[1, i].set_title('')
    axes[1, i].set_xlabel('Churned')

plt.suptitle('Numerical Distributions by Churn Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Step 4 — Skewness and Kurtosis

In [ ]:
print(f'{'Column':<20} {'Skewness':>10} {'Kurtosis':>10} {'Interpretation'}')
print('-' * 65)
for col in num_cols:
    data = df[col].dropna()
    skew = data.skew()
    kurt = data.kurt()
    interp = ''
    if abs(skew) < 0.5:
        interp += 'approx. symmetric'
    elif skew > 0:
        interp += 'right-skewed'
    else:
        interp += 'left-skewed'
    if kurt > 1:
        interp += ', heavy tails'
    print(f'{col:<20} {skew:>10.3f} {kurt:>10.3f}  {interp}')

## 6. Step 5 — Correlation Analysis

In [ ]:
# Fill missing for correlation
df_corr = df.copy()
for col in num_cols:
    df_corr[col] = df_corr[col].fillna(df_corr[col].median())

corr_matrix = df_corr[num_cols + ['churned']].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, mask=mask,
            ax=axes[0], linewidths=0.5, square=True)
axes[0].set_title('Correlation Matrix (lower triangle)')

# Bar chart: correlation with target
corr_target = corr_matrix['churned'].drop('churned').sort_values()
colors = ['#F44336' if v > 0 else '#4CAF50' for v in corr_target.values]
axes[1].barh(corr_target.index, corr_target.values, color=colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Feature Correlation with Target (churned)')
axes[1].set_xlabel('Pearson Correlation')

plt.tight_layout()
plt.show()

print('Strongest positive correlation with churn:', corr_target.idxmax(),
      f'({corr_target.max():.3f})')
print('Strongest negative correlation with churn:', corr_target.idxmin(),
      f'({corr_target.min():.3f})')

## 7. Step 6 — Scatter Matrix (Pairplot)

In [ ]:
plot_cols = ['tenure_months', 'monthly_charge', 'support_calls', 'churned']
df_plot = df_corr[plot_cols].copy()
df_plot['churned'] = df_plot['churned'].map({0: 'Stayed', 1: 'Churned'})

g = sns.pairplot(df_plot, hue='churned', palette={'Stayed': '#4CAF50', 'Churned': '#F44336'},
                 plot_kws={'alpha': 0.4, 's': 15}, diag_kind='kde', corner=True)
g.figure.suptitle('Scatter Matrix — Key Features vs Churn', y=1.01, fontsize=13)
plt.show()

## 8. Step 7 — EDA Report Summary

In [ ]:
print('=' * 60)
print('          EDA REPORT SUMMARY — CUSTOMER CHURN DATASET')
print('=' * 60)
print(f'\nDataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Target:  churned (binary) | Churn rate: {df["churned"].mean():.1%}')

print('\n--- Data Quality ---')
for col in df.columns:
    miss = df[col].isna().sum()
    if miss > 0:
        print(f'  {col}: {miss} missing ({miss/len(df):.1%})')

print('\n--- Key Distributions ---')
for col in num_cols:
    d = df[col].dropna()
    print(f'  {col:<20} mean={d.mean():.1f}  std={d.std():.1f}  skew={d.skew():.2f}')

print('\n--- Churn Rates by Segment ---')
for col in ['contract_type', 'region']:
    best = df.groupby(col)['churned'].mean().idxmin()
    worst = df.groupby(col)['churned'].mean().idxmax()
    print(f'  By {col}: best={best}, worst={worst}')

print('\n--- Top Features Correlated with Churn ---')
top = corr_matrix['churned'].drop('churned').abs().sort_values(ascending=False)
for feat, val in top.items():
    print(f'  {feat:<20} |r| = {val:.3f}')

print('\n--- Recommended Next Steps ---')
print('  1. Impute missing values (median for numeric, mode for categorical)')
print('  2. Encode contract_type and region (one-hot or ordinal)')
print('  3. Feature engineer: charge-per-product, tenure bucket')
print('  4. Address class imbalance if churn rate < 20%')
print('  5. Train baseline classifier (logistic regression, decision tree)')
print('=' * 60)

## Practice Exercises

**Exercise 1 — Deep Dive into a Segment**
Filter the dataset to only Month-to-Month contract customers. Re-run the numerical distribution analysis for this subset and compare the churn rate and feature distributions to the full dataset. What differences do you observe?

**Exercise 2 — Feature Engineering**
Create three new features: `charge_per_product = monthly_charge / num_products`, `tenure_bucket` (bin `tenure_months` into 'New' 0-12, 'Mid' 13-36, 'Long' 37+), and `high_support = support_calls >= 4`. How does churn rate vary across these new features?

**Exercise 3 — Statistical Testing**
Use `scipy.stats.ttest_ind` to test whether the mean `monthly_charge` of churned vs non-churned customers is statistically significantly different. Report the t-statistic, p-value, and your conclusion. Then do the same for `tenure_months`.